# bipgen: development playground

Scratch notebook to try the generator while it is being written. Run it with a Python
environment where bipgen is installed in editable mode (`pip install -e ".[dev]"` from the
repository root). Edits to the package are picked up on re-import thanks to `autoreload`.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import bipgen
from pathlib import Path

ROOT = Path(bipgen.__file__).parents[2]
print(bipgen.__version__, ROOT)

## 1. A specification

Either a YAML file or a plain dictionary in the same layout. Unknown keys are rejected.

In [ ]:
spec = bipgen.load_spec(ROOT / "docs" / "spec_example.yaml")
spec.to_dict()

In [ ]:
# The same thing written as a dictionary, smaller, to iterate quickly
base = {
    "sizes": {"n_bcn": 2000, "n_tgt": 3000},
    "degrees": {"distribution": "lognormal", "bcn": {"median": 15, "sigma": 0.5}, "tgt": {"sigma": 0.6}},
    "edges": {"model": "kernel", "kernel": "gaussian", "scale": 0.12},
    "background": {"model": "configuration", "ratio": 0.05},
}
spec = bipgen.spec_from_dict(base)
spec

## 2. One realization

The seed comes from the specification (default 42) unless given in the call; the seed actually
used is stored in the graph, with the target and expected numbers of edges.

In [ ]:
g = bipgen.generate(spec)            # seed 42
g7 = bipgen.generate(spec, seed=7)  # another realization
print(g)
print(g7)
g.info

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
bipgen.plot_graph(g, ax=axes[0])
bipgen.plot_graph(g7, ax=axes[1]);

In [ ]:
bipgen.plot_degrees(g);

In [ ]:
bipgen.plot_edge_lengths(g);

In [ ]:
# Edge table, one row per edge
g.edges().head()

## 3. Kernel scale and the clipping of probabilities at 1

Sinkhorn matches the target degrees before probabilities are clipped at 1; clipping lowers the
degrees of dense nodes, more so for short kernels. `expected_edges / target_edges` measures it.
For short kernels the potentials keep growing (no convergence, with a warning) even though the
unclipped degrees are matched; when no partner at all is in range, generation stops with an error.

In [ ]:
import warnings

for scale in (0.05, 0.08, 0.12, 0.2, 0.3):
    s = bipgen.spec_from_dict({**base, "edges": {"model": "kernel", "scale": scale}})
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        h = bipgen.generate(s)
    print(f"scale {scale:4.2f}: expected / target = {h.info['expected_edges'] / h.info['target_edges']:.3f}, "
          f"sampled {h.geometric.nnz}, iterations {h.info['sinkhorn_iterations']}, "
          f"converged {h.info['sinkhorn_converged']}, warnings {len(caught)}")
print(caught[-1].message if caught else "")

try:  # far too short: no partner within range for most nodes
    bipgen.generate(bipgen.spec_from_dict({**base, "edges": {"model": "kernel", "scale": 0.005}}))
except ValueError as err:
    print("ValueError:", err)

## 4. An ensemble: fluctuations across seeds

In [ ]:
ensemble = [bipgen.generate(spec, seed=s) for s in range(10)]
n_edges = np.array([h.geometric.nnz for h in ensemble])
print(f"geometric edges over 10 seeds: mean {n_edges.mean():.0f}, sd {n_edges.std():.0f}; "
      f"expected {ensemble[0].info['expected_edges']:.0f} for seed 0")

## 5. k-nearest-neighbour model

Each BCN links to its `k` nearest TGT; no degree section is needed.

In [ ]:
knn = bipgen.spec_from_dict({"sizes": {"n_bcn": 1500, "n_tgt": 1500},
                             "edges": {"model": "knn", "k": 6},
                             "background": {"model": "uniform", "ratio": 0.05}})
gk = bipgen.generate(knn)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
bipgen.plot_graph(gk, ax=axes[0])
bipgen.plot_degrees(gk, axes=axes[1:]);

## 6. 3D

In [ ]:
spec3 = bipgen.spec_from_dict({**base, "dim": 3, "edges": {"model": "kernel", "scale": 0.2}})
g3 = bipgen.generate(spec3)
print(g3, g3.info["expected_edges"] / g3.info["target_edges"])
fig = plt.figure(figsize=(6, 6))
bipgen.plot_graph(g3, ax=fig.add_subplot(projection="3d"), max_edges=1500);

## 7. Spatially correlated density (Metropolis, 2D)

Cell occupancies from a Gamma distribution with coefficient of variation `cv`, rearranged by
simulated annealing until Moran's I at lags 1, 2 matches `correlogram`. The correlogram actually
reached is in `info`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, rho in zip(axes, ([0.0, 0.0], [0.4, 0.2], [0.8, 0.6])):
    s = bipgen.spec_from_dict({**base, "positions": {"method": "metropolis", "density": {
        "n_bins": 20, "occupancy": {"source": "gamma", "cv": 0.8}, "correlogram": rho, "sweeps": 200}}})
    h = bipgen.generate(s)
    bipgen.plot_graph(h, ax=ax, max_edges=0, node_size=2)
    ax.set_title(f"target {rho}, reached {np.round(h.info['correlogram'], 2).tolist()}", fontsize=9)

## 8. Scratch space